**Insights:**

--Every emotionally charged sentence (joy,sadness,anger etc) can be sarcastic or not sarcastic(Purely happiness,sadness,anger etc)

--ML wont be able to detect sarcasam unless whole sentence match with the sentence/phrase in sarcasam dict 

--I would first lable sentence as joy,sadness,anger ---> then will use weak sarcasam dict -->to lable --->manually annotate as much as possible -->as for neutral!=sarcasam we know that

**--no neutral also have to check**

Testing :

Approach 1) when a sentence comes label through emotion
Approach 2 )Based on approach 2 ask model to see if it has touch of sarcasam

Understanding Sarcasam

##  Definition 

**Sarcasm** is when someone says something **positive or neutral on the surface**,
but the **real meaning (intent)** is **negative, mocking, or opposite** of the words used.

It’s a form of **verbal irony** used to **express ridicule, criticism, or humor** — often through **tone, exaggeration, or contrast**.



## **Key Characteristics of Sarcasm**

| Feature                  | Description                                                                      |
| ------------------------ | -------------------------------------------------------------------------------- |
| **Contradiction**        | The literal meaning is opposite to the intended emotion.                         |
| **Tone or exaggeration** | Overly positive words or emojis (“wah”, “amazing 😂”) for bad situations.        |
| **Context dependency**   | Needs situation to understand — “Great!” can be sincere or sarcastic.            |
| **Common markers**       | 😂 😏 🙄 🤦‍♀️ “wah bhai wah”, “great job”, “kya baat hai”, “wah kya timing hai” |

---

##  **Why It’s Hard for Machines**

Sarcasm is **not word-based** — it’s **context-based**:

* Literal meaning ≠ intended meaning.
* Same sentence can be sarcastic or sincere depending on **context**, **tone**, or **emoji**.



The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


In [ ]:
##Create Sarcasam rules to teach to tranformers 
#not in coding ,created for help
# [
#   {
#     "phrase": "wah",
#     "category": "lexical_cue",
#     "rule": "if near [complaint|negative word|fail|problem], label sarcasm"
#   },
#   {
#     "phrase": "bohat acha",
#     "category": "lexical_cue",
#     "rule": "if near negative context or emoji 😒 😂, label sarcasm"
#   },
#   {
#     "phrase": "great job",
#     "category": "lexical_cue",
#     "rule": "if near complaint/negative, label sarcasm"
#   },
#   {
#     "phrase": "😂",
#     "category": "emoji_cue",
#     "rule": "sarcasm if near complaint or contradiction"
#   },
#   {
#     "phrase": "🙄",
#     "category": "emoji_cue",
#     "rule": "sarcasm if near positive words + negative context"
#   }
# ]  
# ## Create Sarcasam weak dictionary for labeling sentences







In [55]:
#Load other Emotions Dictionary and lable by counting max count of emotion_words
import pandas as pd

# Load your CSV
emotion_df = pd.read_csv('/kaggle/input/emotions-word-dictionary/Emotion_dict.csv')

# Build sarcasm dictionary from CSV
sarcasm_df = pd.read_csv('/kaggle/input/emotions-word-dictionary/Emotion_dict.csv')
sarcasm_words = sarcasm_df[sarcasm_df['emotion_category'].isin([sarcasm'])]


print(sarcasm_words)
# Keep only actual emotions, exclude sarcasm
emotion_df = emotion_df[emotion_df['emotion_category'].isin(['joy', 'sadness', 'anger', 'neutral'])]

# Create a dictionary: word -> emotion
word_to_emotion = dict(zip(emotion_df['Word'].str.lower(), emotion_df['emotion_category'].str.lower()))


          Word emotion_category
221        wah          sarcasm
222    shabash          sarcasm
223  mashallah          sarcasm
224  zabardast          sarcasm
225     kamaal          sarcasm
..         ...              ...
375       whiz          sarcasm
376        ace          sarcasm
377      champ          sarcasm
378     winner          sarcasm
379   champion          sarcasm

[159 rows x 2 columns]


In [53]:
#Functions to lable sentences based on emotions excluding sarcasam
def label_sarcasm_phrases(text, sarcasm_words):
    text_lower = text.lower()
    text_clean = text_lower.translate(str.maketrans('', '', string.punctuation))
    
    for phrase in sarcasm_words:
        if phrase in text_clean:
            return 1
    return 0

def label_emotion_with_words(text):
    text_lower = text.lower()
    words = text_lower.split()
    
    emotion_counts = {'joy':0, 'sadness':0, 'anger':0, 'neutral':0  }
    emotion_words = {'joy':[], 'sadness':[], 'anger':[], 'neutral':[]}
    
    for word in words:
        if word in word_to_emotion:
            emotion = word_to_emotion[word]
            emotion_counts[emotion] += 1
            emotion_words[emotion].append(word)
    
    # Choose the dominant emotion
    if max(emotion_counts.values()) > 0:
        dominant_emotion = max(emotion_counts, key=emotion_counts.get)
    else:
        dominant_emotion = 'neutral'
    
    # Return emotion and words that contributed
    return dominant_emotion, emotion_words[dominant_emotion]


In [39]:
#Spelling correction



import pandas as pd
def load_emotion_dictionaries(csv_file_path):
    
    
    df = pd.read_csv(csv_file_path)
    
    # Create emotion dictionaries
    emotion_dicts = {
        'joy': {}, 'sadness': {}, 'anger': {}, 'sarcasm': {}, 'neutral': {}
    }
    
    # Build the dictionaries
    for index, row in df.iterrows():
        emotion = row['emotion_category']
        correct = row['correct_spelling']
        
        # Add all spelling variations
        emotion_dicts[emotion][row['wrong_spelling1']] = correct
        emotion_dicts[emotion][row['wrong_spelling2']] = correct
        emotion_dicts[emotion][row['wrong_spelling3']] = correct
        emotion_dicts[emotion][row['wrong_spelling4']] = correct
        emotion_dicts[emotion][row['wrong_spelling5']] = correct
        emotion_dicts[emotion][correct] = correct
    
    # Create one combined dictionary for easy use
    combined_dict = {}
    for emotion_dict in emotion_dicts.values():
        combined_dict.update(emotion_dict)
    
    return combined_dict


def normalize_text(text, spelling_dict):
    
    
    words = text.split()
    normalized_words = []
    
    for word in words:
        # Convert to lowercase and check in dictionary
        normalized_word = spelling_dict.get(word.lower(), word)
        normalized_words.append(normalized_word)
    
    return ' '.join(normalized_words)


In [1]:

# Example usage
texts = [
    "Wah bhai wah, net gaya phir 😂",
    "Aaj mood acha hai",
    "Kya baat hai, exam fail hogaya!",
    "Bohat mazay ka kaam kiya lol"
]

import pandas as pd

# Step 1: Normalize text using your spelling dictionary
texts_normalised = [normalize_text(text, dictionary) for text in texts]

# Step 2: Create a DataFrame
df = pd.DataFrame(texts_normalised, columns=['text'])

# Step 3: Apply emotion labeling
def apply_labeling(text):
    if not text or pd.isna(text):
        return pd.Series({'emotion':'neutral', 'emotion_words':''})
    # Get dominant emotion and contributing words
    emotion, words = label_emotion_with_words(text)
    return pd.Series({'emotion': emotion, 'emotion_words': ', '.join(words)})

df[['emotion', 'emotion_words']] = df['text'].apply(apply_labeling)

# Step 4: Apply sarcasm labeling (independent from emotions)
df['sarcasm'] = df['text'].apply(lambda x: label_sarcasm(x, sarcasm_words))

# Step 5: View results
print(df)



NameError: name 'normalize_text' is not defined

maza--->mazay  is not mapped rightly so it could not detect happy emotion
emotion word variations not right
acha --> is not included in joy dict
fail---> not in sadness

**overall emotion words and spelling dict is not made rightly**


--here neutral has sarcastic tone so also see neutral


In [ ]:
sarcasm_dict = [
    # Lexical cues / phrases
    {"phrase": "wah", "category": "lexical_cue", "rule": "if near negative context, label sarcasm"},
    {"phrase": "bohat acha", "category": "lexical_cue", "rule": "if near complaint/failure, label sarcasm"},
    {"phrase": "great job", "category": "lexical_cue", "rule": "if near failure/problem, label sarcasm"},
    {"phrase": "kya baat hai", "category": "lexical_cue", "rule": "if near complaint/problem, label sarcasm"},
    {"phrase": "kya scene hai", "category": "lexical_cue", "rule": "if near negative context, label sarcasm"},
    
    # Single words commonly used sarcastically
    {"phrase": "lol", "category": "lexical_cue", "rule": "sarcasm if used in complaint/failure context"},
    {"phrase": "haha", "category": "lexical_cue", "rule": "sarcasm if used in complaint/failure context"},
    {"phrase": "wow", "category": "lexical_cue", "rule": "sarcasm if near negative context"},
    {"phrase": "nice", "category": "lexical_cue", "rule": "sarcasm if near negative context"},
    {"phrase": "good", "category": "lexical_cue", "rule": "sarcasm if near negative context"},
    
    # Emojis commonly indicating sarcasm
    {"phrase": "😂", "category": "emoji_cue", "rule": "sarcasm if near complaint or contradiction"},
    {"phrase": "🙄", "category": "emoji_cue", "rule": "sarcasm if near positive words + negative context"},
    {"phrase": "😒", "category": "emoji_cue", "rule": "sarcasm if near complaint or failure"},
    
    # Social media slang / modern cues
    {"phrase": "pog", "category": "lexical_cue", "rule": "sarcasm if exaggerated context"},
    {"phrase": "based", "category": "lexical_cue", "rule": "sarcasm if exaggerated context"},
    {"phrase": "ratio", "category": "lexical_cue", "rule": "sarcasm if used ironically"},
    {"phrase": "mid", "category": "lexical_cue", "rule": "sarcasm if used ironically"},
    {"phrase": "sus", "category": "lexical_cue", "rule": "sarcasm if used ironically"}
]


#make sepaarte file for sarcasam dict 
#make separte for all emotiosn so es to modify
#weak labelling 

#scrap after sarcasam rightly labelled


In [ ]:
import string

def label_sarcasm_weak(text, sarcasm_dict):
    # Normalize text
    text_clean = text.lower().translate(str.maketrans('', '', string.punctuation))
    
    # Check each phrase in sarcasm_dict
    for item in sarcasm_dict:
        if item["phrase"].lower() in text_clean:
            return 1  # Sarcasm detected
    return 0


In [ ]:
#Scrap data from reddit 


#do spelling correction 



#lable data 


# Manually lable 


#download final training data